# Stage 2: Selective Contraction + SMBO + GINE

**Pipeline**: crop → protect → merge_and_cut_protected → default deletion → **selective contraction** → GINE

**Key idea**: aggressively merge background supernodes, preserve tumor candidates. SMBO optimizes all parameters jointly.

**Target**: 172K nodes → ~10K nodes → GINE Dice ≥ 0.891

In [1]:
import numpy as np
import os, re, time, json, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, BatchNorm
from scipy.ndimage import binary_dilation
import optuna
import fastloops

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_ROOT = "/scratch/ud3d4/acm_data/Data"
RESULTS_DIR = "/home/ud3d4/Desktop/SWOG/results/stage2_selective"
os.makedirs(RESULTS_DIR, exist_ok=True)

HU_MIN, HU_MAX = -50, 250
np.random.seed(42); torch.manual_seed(42)

def pr(msg=""): print(msg, flush=True)

# ---- Data loading ----
def load_and_convert(vid):
    ct = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
    seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
    ct_u8 = np.clip(ct, HU_MIN, HU_MAX)
    ct_u8 = ((ct_u8 - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    return ct, seg, np.ascontiguousarray(ct_u8[..., np.newaxis])

def discover_volumes():
    vids = []
    for f in sorted(os.listdir(os.path.join(DATA_ROOT, "ct"))):
        m = re.match(r"volume-(\d+)\.npy", f)
        if m:
            vid = int(m.group(1))
            seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy"))
            if (seg == 2).sum() > 0: vids.append(vid)
    return sorted(vids)

def bbox_from_mask(mask, margin=32):
    coords = np.argwhere(mask)
    lo = np.maximum(coords.min(0) - margin, 0)
    hi = np.minimum(coords.max(0) + 1 + margin, mask.shape)
    return tuple(slice(int(lo[i]), int(hi[i])) for i in range(3))

all_vids = discover_volumes()
perm = np.random.permutation(len(all_vids))
n_tr = int(0.7 * len(all_vids)); n_va = int(0.15 * len(all_vids))
train_ids = sorted([all_vids[i] for i in perm[:n_tr]])
val_ids = sorted([all_vids[i] for i in perm[n_tr:n_tr+n_va]])
test_ids = sorted([all_vids[i] for i in perm[n_tr+n_va:]])
pr(f"Found {len(all_vids)} volumes. Split: {len(train_ids)}/{len(val_ids)}/{len(test_ids)}")
pr(f"Device: {'cuda:0' if torch.cuda.is_available() else 'cpu'}")

/home/ud3d4/.conda/envs/llmft/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 118 volumes. Split: 82/17/19


Device: cuda:0


## Stage 1: Crop + Protect + Coarsen + Delete (our v2 pipeline)

In [2]:
def make_protection(ct_u8_crop, organ_crop, std_mult=1.5, dilate_iter=2):
    liver_vals = ct_u8_crop[..., 0][organ_crop]
    candidate = organ_crop & (ct_u8_crop[..., 0] < liver_vals.mean() - std_mult * liver_vals.std())
    if dilate_iter > 0:
        candidate = binary_dilation(candidate, iterations=dilate_iter)
    return candidate.astype(np.uint8)

def build_stage1(vid, psi=5, alpha=25, std_mult=1.5, dilate_iter=2):
    """Full Stage 1: crop → protect → coarsen → delete. Returns raw graph data."""
    ct_raw, seg, ct_u8_full = load_and_convert(vid)
    organ_mask = (seg == 1) | (seg == 2)
    slc = bbox_from_mask(organ_mask)
    ct_crop = ct_raw[slc]; seg_crop = seg[slc]; organ_crop = organ_mask[slc]
    ct_u8_crop = np.clip(ct_crop, HU_MIN, HU_MAX)
    ct_u8_crop = ((ct_u8_crop - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    ct_u8_crop = np.ascontiguousarray(ct_u8_crop[..., np.newaxis])
    
    protect = make_protection(ct_u8_crop, organ_crop, std_mult, dilate_iter)
    nf, ei, ef, labels, adj = fastloops.merge_and_cut_protected(
        ct_u8_crop, np.ascontiguousarray(protect),
        merge_distance=psi, cut_distance=alpha, connectivity="faces")
    
    labels_np = np.asarray(labels)
    return dict(
        vid=vid, raw_nf=np.asarray(nf), raw_ei=np.asarray(ei), raw_ef=np.asarray(ef),
        labels=labels_np, seg=seg_crop, ct_u8=ct_u8_crop, protect=protect, slc=slc)

# ---- Feature extraction (Luke's 7 node + 12 edge) ----
def _layout(C=1):
    return dict(area=0, s=[1, 2, 3],
                cov=[(4,0,0),(5,1,1),(6,2,2),(7,0,1),(8,0,2),(9,1,2)],
                chan0=10, boundary=10+C+6, D=3)

def node_invariants(nf, C=1, eps=1e-6):
    f = nf.astype(np.float64); L = _layout(C); N = f.shape[0]
    V = f[:, L["area"]]; Vsafe = np.maximum(V, 1.0)
    mc = np.stack([f[:, c] for c in L["s"]], axis=1) / Vsafe[:, None]
    cov = np.zeros((N, 3, 3))
    for col, i, j in L["cov"]:
        cij = f[:, col] / Vsafe - mc[:, i] * mc[:, j]
        cov[:, i, j] = cij; cov[:, j, i] = cij
    w, vec = np.linalg.eigh(cov); w = np.clip(w, 0.0, None)
    principal = vec[..., -1]; trace = w.sum(1); degen = trace < eps
    denom = w[:, 2] + eps
    lin = (w[:, 2] - w[:, 1]) / denom
    pla = (w[:, 1] - w[:, 0]) / denom
    sph = w[:, 0] / denom
    shape = np.stack([lin, pla, sph], axis=1); shape[degen] = 0.0
    line_like = np.where(degen, 0.0, lin)
    chan = f[:, L["chan0"]:L["chan0"]+C] / Vsafe[:, None] / 255.0
    comp = f[:, L["boundary"]] / np.power(Vsafe, 2.0/3.0)
    return dict(V=V, surface=f[:, L["boundary"]], centroid=mc, eig=w,
                shape=shape, line_like=line_like, principal=principal,
                chan=chan, compactness=comp, degenerate=degen.astype(np.float64))

def make_node_features(nf, C=1):
    inv = node_invariants(nf, C)
    x = np.column_stack([
        np.log1p(inv["V"]), inv["chan"][:, 0],
        inv["shape"][:, 0], inv["shape"][:, 1], inv["shape"][:, 2],
        inv["compactness"], inv["degenerate"],
    ]).astype(np.float32)
    for col in range(x.shape[1]):
        mu, sig = x[:, col].mean(), x[:, col].std()
        if sig > 1e-8: x[:, col] = (x[:, col] - mu) / sig
        else: x[:, col] = 0.0
    return x

def make_edge_features(nf, ei, ef, C=1, eps=1e-6):
    inv = node_invariants(nf, C, eps)
    a, b = ei[0].astype(np.int64), ei[1].astype(np.int64)
    e = ef.astype(np.float64); blsafe = np.maximum(e[:, 0], 1.0)
    Va, Vb = inv["V"][a], inv["V"][b]
    sc = np.abs(Va - Vb) / (Va + Vb + eps)
    bfa = e[:, 0] / (inv["surface"][a] + eps)
    bfb = e[:, 0] / (inv["surface"][b] + eps)
    mua, mub = inv["chan"][a], inv["chan"][b]
    mc = np.abs(mua - mub) / (mua + mub + eps)
    sd = np.abs(inv["shape"][a] - inv["shape"][b])
    aa = (np.abs(np.sum(inv["principal"][a] * inv["principal"][b], axis=1))
          * np.minimum(inv["line_like"][a], inv["line_like"][b]))
    bc = (e[:, 1] / blsafe) / 255.0
    # Log-ratios (antisymmetric)
    Vas, Vbs = np.maximum(Va, 1.0), np.maximum(Vb, 1.0)
    lr_s = np.log(Vas / Vbs)
    ca, cb = np.maximum(mua, eps), np.maximum(mub, eps)
    lr_i = np.log(ca / cb)
    cpa, cpb = np.maximum(inv["compactness"][a], eps), np.maximum(inv["compactness"][b], eps)
    lr_c = np.log(cpa / cpb)
    cols = [sc[:,None], bfa[:,None], bfb[:,None],
            mc if mc.ndim>1 else mc[:,None], sd, aa[:,None], bc[:,None],
            lr_s[:,None], lr_i[:,None] if lr_i.ndim==1 else lr_i, lr_c[:,None]]
    return np.concatenate(cols, axis=1).astype(np.float32)

pr("Stage 1 + feature extraction defined.")

Stage 1 + feature extraction defined.


## Stage 2: Selective Contraction

Aggressively merge unprotected (background) supernodes. Preserve protected (tumor candidate) supernodes. Never merge across boundary.

In [3]:
def selective_contract(d, bg_threshold=10, prot_threshold=3, n_rounds=3):
    raw_nf = d["raw_nf"].copy(); raw_ei = d["raw_ei"].copy()
    raw_ef = d["raw_ef"].copy(); labels_np = d["labels"]; protect = d["protect"]
    n_nodes = raw_nf.shape[0]
    if n_nodes == 0 or raw_ei.shape[1] == 0:
        return raw_nf, raw_ei, raw_ef, np.arange(n_nodes), np.zeros(n_nodes, dtype=bool)
    flat_labels = labels_np.ravel(); flat_protect = protect.ravel().astype(bool)
    valid = flat_labels >= 0; max_id = int(flat_labels[valid].max()) if valid.any() else -1
    node_protected = np.zeros(n_nodes, dtype=bool)
    if max_id >= 0:
        pc = np.bincount(flat_labels[valid & flat_protect], minlength=max_id+1)
        node_protected[:min(n_nodes, len(pc))] = pc[:n_nodes] > 0
    L = _layout(); cumulative_map = np.arange(n_nodes, dtype=np.int64)
    for rnd in range(n_rounds):
        n_cur = raw_nf.shape[0]
        if n_cur == 0 or raw_ei.shape[1] == 0: break
        area = raw_nf[:, L["area"]].astype(np.float64)
        mean_int = raw_nf[:, L["chan0"]].astype(np.float64) / np.maximum(area, 1.0)
        src = raw_ei[0].astype(np.int64); dst = raw_ei[1].astype(np.int64)
        parent = np.arange(n_cur, dtype=np.int64)
        def find(x):
            while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
            return x
        merges = 0
        for idx in range(len(src)):
            a, b = int(src[idx]), int(dst[idx])
            if a >= n_cur or b >= n_cur: continue
            ra, rb = find(a), find(b)
            if ra == rb: continue
            if node_protected[ra] != node_protected[rb]: continue
            diff = abs(mean_int[ra] - mean_int[rb])
            th = prot_threshold if node_protected[ra] else bg_threshold
            if diff <= th: parent[rb] = ra; merges += 1
        if merges == 0: break
        roots = np.array([find(i) for i in range(n_cur)])
        unique_roots, inverse = np.unique(roots, return_inverse=True); new_n = len(unique_roots)
        new_nf = np.zeros((new_n, raw_nf.shape[1]), dtype=np.uint64)
        new_prot = np.zeros(new_n, dtype=bool)
        for i in range(n_cur):
            j = inverse[i]; new_nf[j, :11] += raw_nf[i, :11].astype(np.uint64)
            new_nf[j, 17] += raw_nf[i, 17].astype(np.uint64)
            for c in [11,13,15]:
                if new_nf[j,c]==0 or raw_nf[i,c]<new_nf[j,c]: new_nf[j,c]=raw_nf[i,c]
            for c in [12,14,16]:
                if raw_nf[i,c]>new_nf[j,c]: new_nf[j,c]=raw_nf[i,c]
            if node_protected[i]: new_prot[j] = True
        for idx in range(len(src)):
            a,b = int(src[idx]),int(dst[idx])
            if a>=n_cur or b>=n_cur: continue
            ja,jb = inverse[roots[a]], inverse[roots[b]]
            if ja==jb:
                v = int(new_nf[ja,17]) - 2*int(raw_ef[idx,0]); new_nf[ja,17] = max(0,v)
        edge_dict = {}
        for idx in range(len(src)):
            a,b = int(src[idx]),int(dst[idx])
            if a>=n_cur or b>=n_cur: continue
            ja,jb = inverse[roots[a]], inverse[roots[b]]
            if ja==jb: continue
            key = (min(ja,jb),max(ja,jb))
            if key not in edge_dict: edge_dict[key] = np.zeros(raw_ef.shape[1], dtype=np.float64)
            edge_dict[key] += raw_ef[idx].astype(np.float64)
        if edge_dict:
            keys = sorted(edge_dict.keys())
            new_ei = np.array([[k[0] for k in keys],[k[1] for k in keys]], dtype=np.int64)
            new_ef = np.array([edge_dict[k] for k in keys], dtype=np.uint64)
        else:
            new_ei = np.zeros((2,0),dtype=np.int64); new_ef = np.zeros((0,raw_ef.shape[1]),dtype=np.uint64)
        new_cum = np.zeros(len(cumulative_map), dtype=np.int64)
        for orig in range(len(cumulative_map)):
            prev = cumulative_map[orig]
            new_cum[orig] = inverse[roots[prev]] if prev < n_cur else 0
        cumulative_map = new_cum
        raw_nf=new_nf; raw_ei=new_ei; raw_ef=new_ef; node_protected=new_prot; n_nodes=new_n
        pr(f"    Round {rnd+1}: {merges} merges -> {new_n:,} nodes, {len(edge_dict):,} edges")
    return raw_nf, raw_ei, raw_ef, cumulative_map, node_protected

def build_pyg_from_contracted(raw_nf, raw_ei, raw_ef, mapping, d, overlap_th=0.10):
    n_nodes = raw_nf.shape[0]
    if n_nodes == 0: return None
    x = make_node_features(raw_nf)
    if raw_ei.shape[1] > 0:
        ea = make_edge_features(raw_nf, raw_ei, raw_ef)
        ei_fwd = torch.tensor(raw_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr = torch.tensor(np.concatenate([ea, ea]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2,0), dtype=torch.long)
        edge_attr = torch.zeros((0,10), dtype=torch.float32)
    flat = d["labels"].ravel(); valid = flat >= 0
    gt = (d["seg"].ravel() == 2).astype(np.float64)
    n_fg = np.zeros(n_nodes, dtype=np.float64); n_bg = np.zeros(n_nodes, dtype=np.float64)
    if valid.any():
        mx = int(flat[valid].max())
        ofg = np.bincount(flat[valid], weights=gt[valid], minlength=mx+1)
        otot = np.bincount(flat[valid], minlength=mx+1); obg = otot - ofg
        for oid in range(min(len(mapping), len(ofg))):
            nid = mapping[oid]
            if nid < n_nodes: n_fg[nid] += ofg[oid]; n_bg[nid] += obg[oid]
    overlap = n_fg / np.maximum(n_fg + n_bg, 1)
    y = (overlap >= overlap_th).astype(np.int64)
    return Data(x=torch.tensor(x, dtype=torch.float32), edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor(y, dtype=torch.long),
                n_fg=torch.tensor(n_fg, dtype=torch.float32), n_bg=torch.tensor(n_bg, dtype=torch.float32))

def oracle_dice_contracted(data):
    fg = data.n_fg.detach().cpu().numpy(); bg = data.n_bg.detach().cpu().numpy()
    gt_total = fg.sum()
    if gt_total == 0: return 0.0
    maj = fg > bg; tp = fg[maj].sum(); fp_bg = bg[maj].sum()
    return float(2 * tp / (2 * tp + fp_bg + (gt_total - tp) + 1e-8))

pr("Stage 2 selective contraction defined.")

Stage 2 selective contraction defined.


## SMBO Phase 1: Optimize graph params against oracle Dice (fast, no GINE)

In [4]:
# Pre-build Stage 1 graphs for SMBO volumes
SMBO_VIDS = val_ids
pr(f"Building Stage 1 graphs for {len(SMBO_VIDS)} SMBO volumes...")
smbo_stage1 = {}
for vid in SMBO_VIDS:
    t0 = time.time()
    d = build_stage1(vid)
    smbo_stage1[vid] = d
    n = d["raw_nf"].shape[0]
    pr(f"  vol-{vid}: {n:,} nodes  ({time.time()-t0:.1f}s)")
pr("Stage 1 graphs ready for SMBO.")


def smbo_objective(trial):
    """Optuna objective: maximize mean oracle Dice on contracted val graphs."""
    psi = trial.suggest_int("psi", 2, 15)
    alpha = trial.suggest_int("alpha", psi * 2, psi * 10)
    std_mult = trial.suggest_float("std_mult", 0.5, 2.0)
    dilate_iter = trial.suggest_int("dilate_iter", 0, 3)
    bg_threshold = trial.suggest_int("bg_threshold", 3, 30)
    prot_threshold = trial.suggest_int("prot_threshold", 1, 15)
    n_rounds = trial.suggest_int("n_rounds", 1, 5)
    
    oracle_dices = []
    total_nodes = []
    
    for vid in SMBO_VIDS:
        try:
            d = build_stage1(vid, psi=psi, alpha=alpha, 
                           std_mult=std_mult, dilate_iter=dilate_iter)
            c_nf, c_ei, c_ef, mapping, _ = selective_contract(
                d, bg_threshold=bg_threshold, prot_threshold=prot_threshold, n_rounds=n_rounds)
            data = build_pyg_from_contracted(c_nf, c_ei, c_ef, mapping, d)
            if data is None:
                continue
            od = oracle_dice_contracted(data)
            oracle_dices.append(od)
            total_nodes.append(data.num_nodes)
        except Exception as e:
            continue  # skip failed volumes
    
    if len(oracle_dices) < len(SMBO_VIDS) // 2:
        return 0.0  # too many failures
    
    mean_oracle = np.mean(oracle_dices)
    mean_nodes = np.mean(total_nodes)
    
    node_penalty = max(0, (mean_nodes - 50000) / 50000) * 0.1
    
    trial.set_user_attr("mean_nodes", float(mean_nodes))
    trial.set_user_attr("mean_oracle", float(mean_oracle))
    trial.set_user_attr("n_success", len(oracle_dices))
    
    return mean_oracle - node_penalty


pr("SMBO objective defined. Starting optimization...")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(smbo_objective, n_trials=40, timeout=3600, catch=(Exception,))

pr(f"\nBest trial: {study.best_trial.number}")
pr(f"  Value (oracle - penalty): {study.best_value:.4f}")
pr(f"  Params: {study.best_params}")
pr(f"  Mean nodes: {study.best_trial.user_attrs.get('mean_nodes', '?')}")
pr(f"  Mean oracle: {study.best_trial.user_attrs.get('mean_oracle', '?')}")

BEST = study.best_params
with open(os.path.join(RESULTS_DIR, "smbo_results.json"), "w") as f:
    json.dump({"best_params": BEST, "best_value": study.best_value,
               "n_trials": len(study.trials),
               "best_attrs": study.best_trial.user_attrs}, f, indent=2)

Building Stage 1 graphs for 17 SMBO volumes...


  vol-1: 30,814 nodes  (0.2s)


  vol-29: 115,447 nodes  (0.6s)


  vol-33: 106,255 nodes  (0.5s)


  vol-40: 132,829 nodes  (0.6s)


...


Best trial: 35


  Value (oracle - penalty): 0.8506


  Params: {'psi': 9, 'alpha': 83, 'std_mult': 1.8622670470989797, 'dilate_iter': 1, 'bg_threshold': 5, 'prot_threshold': 9, 'n_rounds': 1}


  Mean nodes: 52809.17647058824


  Mean oracle: 0.8562253134413889


## Phase 2: Build all graphs with best params + Train GINE

In [5]:
# Build all 118 graphs with SMBO-optimized params
pr(f"Building all graphs with SMBO params: {BEST}")

all_graphs = {}; all_mappings = {}; all_stage1 = {}
t0 = time.time()

for vid in train_ids + val_ids + test_ids:
    d = build_stage1(vid, psi=BEST["psi"], alpha=BEST["alpha"],
                     std_mult=BEST["std_mult"], dilate_iter=BEST["dilate_iter"])
    all_stage1[vid] = d
    
    c_nf, c_ei, c_ef, mapping, _ = selective_contract(
        d, bg_threshold=BEST["bg_threshold"],
        prot_threshold=BEST["prot_threshold"], n_rounds=BEST["n_rounds"])
    
    data = build_pyg_from_contracted(c_nf, c_ei, c_ef, mapping, d)
    if data is None:
        pr(f"  vol-{vid}: SKIPPED (empty graph)")
        continue
    
    all_graphs[vid] = data
    all_mappings[vid] = mapping
    
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    n_tu = int((data.y == 1).sum())
    od = oracle_dice_contracted(data)
    pr(f"  vol-{vid:>3d} [{split:>5s}]: {data.num_nodes:>7,} nodes ({n_tu:>5,} tu) "
       f"{data.num_edges:>8,} edges  oracle={od:.4f}")

pr(f"\nBuilt {len(all_graphs)} graphs in {time.time()-t0:.1f}s")
nodes_list = [g.num_nodes for g in all_graphs.values()]
pr(f"Nodes: mean={np.mean(nodes_list):,.0f}, median={np.median(nodes_list):,.0f}, "
   f"max={max(nodes_list):,}, min={min(nodes_list):,}")
oracles = [oracle_dice_contracted(g) for g in all_graphs.values()]
pr(f"Oracle Dice: mean={np.mean(oracles):.4f}")

Building all graphs with SMBO params: {'psi': 9, 'alpha': 83, 'std_mult': 1.8622670470989797, 'dilate_iter': 1, 'bg_threshold': 5, 'prot_threshold': 9, 'n_rounds': 1}


    Round 1: 805 merges -> 11,283 nodes, 25,539 edges


  vol-  0 [train]:  11,283 nodes (  118 tu)   51,078 edges  oracle=0.8733


    Round 1: 3381 merges -> 68,156 nodes, 156,777 edges


  vol-  3 [train]:  68,156 nodes (   99 tu)  313,554 edges  oracle=0.7936


...

    Round 1: 10080 merges -> 102,101 nodes, 273,199 edges


  vol-130 [ test]: 102,101 nodes (47,475 tu)  546,398 edges  oracle=0.8773



Built 118 graphs in 250.4s


Nodes: mean=55,080, median=43,998, max=281,538, min=7,052


Oracle Dice: mean=0.8270


In [6]:
# ---- GINE model ----
class GINE(nn.Module):
    def __init__(self, nd, ed, h=128):
        super().__init__()
        self.ep = nn.Linear(ed, h)
        def mlp(d): return nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Linear(h, h))
        self.c1 = GINEConv(mlp(nd), edge_dim=h); self.b1 = BatchNorm(h)
        self.c2 = GINEConv(mlp(h), edge_dim=h); self.b2 = BatchNorm(h)
        self.c3 = GINEConv(mlp(h), edge_dim=h); self.b3 = BatchNorm(h)
        self.head = nn.Linear(h, 2)
    def forward(self, x, ei, ea):
        if ea is not None and ea.numel() > 0: ea = self.ep(ea)
        else:
            n = x.size(0); ei = torch.stack([torch.arange(n, device=x.device)]*2)
            ea = torch.zeros(n, self.ep.out_features, device=x.device)
        x = F.relu(self.b1(self.c1(x, ei, ea)))
        x = F.relu(self.b2(self.c2(x, ei, ea)))
        x = F.relu(self.b3(self.c3(x, ei, ea)))
        return self.head(x)

# ---- Train ----
device = "cuda:0" if torch.cuda.is_available() else "cpu"
MAX_NODES = 500_000
trainable = [v for v in train_ids if v in all_graphs and all_graphs[v].num_nodes <= MAX_NODES]
val_usable = [v for v in val_ids if v in all_graphs]
pr(f"Trainable: {len(trainable)}/{len(train_ids)}, Val: {len(val_usable)}/{len(val_ids)}")

total_pos = sum(int((all_graphs[v].y == 1).sum()) for v in trainable)
total_neg = sum(int((all_graphs[v].y == 0).sum()) for v in trainable)
ratio = total_neg / max(total_pos, 1)
eff = min(np.sqrt(ratio), 30.0)
cw = torch.tensor([1.0, eff], dtype=torch.float32).to(device)
pr(f"Class weight: [1.0, {eff:.1f}] (ratio: {ratio:.0f}:1)")

nd = all_graphs[trainable[0]].x.shape[1]
ed = all_graphs[trainable[0]].edge_attr.shape[1] if all_graphs[trainable[0]].edge_attr.numel() > 0 else 10
model = GINE(nd, ed).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

PATIENCE = 15; EPOCHS = 200; VAL_EVERY = 3
best_dice, best_state, wait = -1.0, None, 0
history = {"train_loss": [], "val_dice": []}

for epoch in range(1, EPOCHS+1):
    model.train(); eloss = 0.0; proc = 0
    for vid in np.random.permutation(trainable):
        try:
            g = all_graphs[vid].to(device); opt.zero_grad()
            logits = model(g.x, g.edge_index, g.edge_attr)
            loss = F.cross_entropy(logits, g.y, weight=cw)
            loss.backward(); opt.step(); eloss += loss.item(); proc += 1
            del g, logits, loss
        except torch.cuda.OutOfMemoryError:
            try: del g
            except: pass
            torch.cuda.empty_cache(); continue
        torch.cuda.empty_cache()
    if proc == 0: pr(f"Epoch {epoch}: all OOM"); break
    ml = eloss / proc; history["train_loss"].append(ml)
    
    if epoch % VAL_EVERY == 0 or epoch <= 3:
        model.eval(); tp = fp = fn = 0
        with torch.no_grad():
            for vid in val_usable:
                g = all_graphs[vid]; d = all_stage1[vid]; mapping = all_mappings[vid]
                try:
                    gd = g.to(device)
                    preds = model(gd.x, gd.edge_index, gd.edge_attr).argmax(1).cpu().numpy()
                    del gd; torch.cuda.empty_cache()
                except:
                    torch.cuda.empty_cache()
                    preds = model.cpu()(g.x, g.edge_index, g.edge_attr).argmax(1).numpy()
                    model.to(device)
                # Lift: original supernode → contracted node → prediction → voxel
                flat = d["labels"].ravel(); valid = flat >= 0
                if not valid.any(): continue
                mid = int(flat[valid].max())
                lut = np.zeros(mid+1, dtype=np.int8)
                for oid in range(min(len(mapping), mid+1)):
                    nid = mapping[oid]
                    if nid < len(preds): lut[oid] = preds[nid]
                pm = np.where(valid, lut[flat], 0).reshape(d["labels"].shape).astype(bool)
                gm = d["seg"] == 2; inter = int((pm & gm).sum())
                tp += inter; fp += int(pm.sum()) - inter; fn += int(gm.sum()) - inter
        
        vd = 2*tp/(2*tp+fp+fn+1e-8); history["val_dice"].append(vd)
        if vd > best_dice:
            best_dice = vd; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0; m = " *"
        else: wait += 1; m = ""
        pr(f"  Epoch {epoch:3d}  loss={ml:.4f}  val_dice={vd:.4f}  ({proc} vols){m}")
        if wait >= PATIENCE: pr(f"  Early stop, best={best_dice:.4f}"); break
    else: pr(f"  Epoch {epoch:3d}  loss={ml:.4f}  ({proc} vols)")

if best_state: model.load_state_dict(best_state)
pr(f"\nBest val Dice: {best_dice:.4f}")
torch.save(best_state or model.state_dict(), os.path.join(RESULTS_DIR, "model.pt"))

Trainable: 82/82, Val: 17/17


Class weight: [1.0, 3.2] (ratio: 10:1)


  Epoch   1  loss=0.5265  val_dice=0.0000  (82 vols) *


  Epoch   2  loss=0.3785  val_dice=0.0000  (82 vols)


  Epoch   3  loss=0.3631  val_dice=0.1033  (82 vols) *


...

  Epoch  88  loss=0.3417  (82 vols)


  Epoch  89  loss=0.3348  (82 vols)


  Epoch  90  loss=0.3398  val_dice=0.0000  (82 vols)


  Early stop, best=0.1312



Best val Dice: 0.1312


## Final evaluation

In [7]:
model.eval(); results = []
for vid in sorted(all_graphs.keys()):
    g = all_graphs[vid]; d = all_stage1[vid]; mapping = all_mappings[vid]
    with torch.no_grad():
        try:
            gd = g.to(device)
            preds = model(gd.x, gd.edge_index, gd.edge_attr).argmax(1).cpu().numpy()
            del gd; torch.cuda.empty_cache()
        except:
            torch.cuda.empty_cache()
            preds = model.cpu()(g.x, g.edge_index, g.edge_attr).argmax(1).numpy()
            model.to(device)
    flat = d["labels"].ravel(); valid = flat >= 0
    pm = np.zeros(d["labels"].shape, dtype=bool)
    if valid.any():
        mid = int(flat[valid].max())
        lut = np.zeros(mid+1, dtype=np.int8)
        for oid in range(min(len(mapping), mid+1)):
            nid = mapping[oid]
            if nid < len(preds): lut[oid] = preds[nid]
        pm = np.where(valid, lut[flat], 0).reshape(d["labels"].shape).astype(bool)
    gm = d["seg"] == 2; inter = int((gm & pm).sum())
    dice = 2.0*inter/(gm.sum()+pm.sum()+1e-8)
    rec = inter/(gm.sum()+1e-8)
    prec = inter/(pm.sum()+1e-8) if pm.sum()>0 else 0.0
    od = oracle_dice_contracted(g)
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    results.append(dict(vid=vid, split=split, dice=float(dice), recall=float(rec),
                        precision=float(prec), oracle=float(od), nodes=g.num_nodes))

pr(f"\n{'='*80}")
pr(f"  RESULTS: Selective Contraction + SMBO + GINE")
pr(f"{'='*80}")
pr(f"  SMBO params: {BEST}")
pr(f"  {'Split':>5s}  {'Dice':>8s}  {'Recall':>8s}  {'Prec':>8s}  {'Oracle':>8s}  {'Nodes':>8s}  {'N':>4s}")
pr(f"  {'-'*55}")
for split in ["train", "val", "test"]:
    s = [r for r in results if r["split"] == split]
    if s:
        pr(f"  {split:>5s}  {np.mean([r['dice'] for r in s]):8.4f}  "
           f"{np.mean([r['recall'] for r in s]):8.4f}  "
           f"{np.mean([r['precision'] for r in s]):8.4f}  "
           f"{np.mean([r['oracle'] for r in s]):8.4f}  "
           f"{np.mean([r['nodes'] for r in s]):8.0f}  {len(s):4d}")

pr(f"\n  Best val Dice: {best_dice:.4f}")
pr(f"  Paper target: 0.891 +/- 0.007")
pr(f"\n  Previous best (v2 S2, 172K nodes): val 0.48, test 0.32")

# Save
with open(os.path.join(RESULTS_DIR, "final_results.json"), "w") as f:
    json.dump(dict(smbo_params=BEST, best_val_dice=float(best_dice),
                   epochs=len(history["train_loss"]), history=history, results=results), f, indent=2)
pr(f"  Saved to {RESULTS_DIR}/")

  RESULTS: Selective Contraction + SMBO + GINE


  SMBO params: {'psi': 9, 'alpha': 83, 'std_mult': 1.8622670470989797, 'dilate_iter': 1, 'bg_threshold': 5, 'prot_threshold': 9, 'n_rounds': 1}


  Split      Dice    Recall      Prec    Oracle     Nodes     N


  -------------------------------------------------------


  train    0.0897    0.0956    0.2233    0.8150     54034    82


...

   test    0.0495    0.0466    0.2042    0.8526     61627    19



  Best val Dice: 0.1312


  Paper target: 0.891 +/- 0.007



  Previous best (v2 S2, 172K nodes): val 0.48, test 0.32


  Saved to /home/ud3d4/Desktop/SWOG/results/stage2_selective/
